# Module 3: Policy Area Network Topology

**Question:** How do policy areas relate to each other through shared sponsors?

Analyses:
1. Policy co-occurrence graph — shared sponsors between policy domains
2. Taxonomy co-occurrence — which harms are legislated alongside which strategies?
3. Policy area centrality — which domains are most connected?
4. Isolated vs. integrated domains

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath(".")))

import pandas as pd
import numpy as np
import networkx as nx
import plotly.express as px
import plotly.graph_objects as go
from collections import defaultdict

import analysis_utils as au

docs_df = au.load_comprehensive_df()
cosp_df = au.load_cosponsors_df()

print(f"Documents: {len(docs_df)}, Cosponsor records: {len(cosp_df)}")
print(f"Policy areas: {docs_df['Policy_Area'].nunique()}")

## 3.1 Policy Co-occurrence Graph

Nodes = policy areas, edge weight = number of sponsors active in both areas. This reveals which policy domains attract overlapping legislative coalitions.

In [ ]:
# Build policy co-occurrence graph
G_pol = au.build_policy_cooccurrence_graph(docs_df, cosp_df)

print(f"Policy co-occurrence graph: {G_pol.number_of_nodes()} nodes, {G_pol.number_of_edges()} edges")
print(f"\nPolicy areas by sponsor count:")
for node, data in sorted(G_pol.nodes(data=True), key=lambda x: x[1].get("sponsor_count", 0), reverse=True):
    print(f"  {node}: {data.get('sponsor_count', 0)} sponsors")

# Network layout
pos = nx.spring_layout(G_pol, weight="weight", seed=42, k=2.0)

# Prepare Plotly network visualization
edge_x, edge_y, edge_widths, edge_hover = [], [], [], []
for u, v, d in G_pol.edges(data=True):
    x0, y0 = pos[u]
    x1, y1 = pos[v]
    edge_x += [x0, x1, None]
    edge_y += [y0, y1, None]
    edge_widths.append(d["weight"])
    edge_hover.append(f"{u} ↔ {v}: {d['weight']} shared sponsors")

# Normalize edge widths for display
max_w = max(edge_widths) if edge_widths else 1

node_x = [pos[n][0] for n in G_pol.nodes()]
node_y = [pos[n][1] for n in G_pol.nodes()]
node_text = [f"{n}<br>{G_pol.nodes[n].get('sponsor_count', 0)} sponsors<br>Degree: {G_pol.degree(n, weight='weight')}" for n in G_pol.nodes()]
node_size = [max(10, G_pol.nodes[n].get("sponsor_count", 5)) / 3 for n in G_pol.nodes()]
node_labels = list(G_pol.nodes())

fig_net = go.Figure()

# Edges with varying width
for u, v, d in G_pol.edges(data=True):
    x0, y0 = pos[u]
    x1, y1 = pos[v]
    w = max(0.5, (d["weight"] / max_w) * 8)
    fig_net.add_trace(go.Scatter(
        x=[x0, x1], y=[y0, y1], mode="lines",
        line=dict(width=w, color="rgba(150,150,150,0.4)"),
        hoverinfo="text", text=f"{u} ↔ {v}: {d['weight']} shared sponsors",
        showlegend=False,
    ))

# Nodes
fig_net.add_trace(go.Scatter(
    x=node_x, y=node_y, mode="markers+text",
    marker=dict(size=node_size, color="#1f77b4", line=dict(width=1, color="white")),
    text=node_labels, textposition="top center", textfont=dict(size=9),
    hovertext=node_text, hoverinfo="text",
    showlegend=False,
))

fig_net.update_layout(
    title="Policy Area Co-occurrence Network (edge width = shared sponsors)",
    template="plotly_white", height=600, width=800,
    xaxis=dict(visible=False), yaxis=dict(visible=False),
)
fig_net.show()

## 3.2 Policy Area Centrality

Which policy areas are most "connected" through shared legislative activity? Eigenvector centrality captures influence — a highly central policy area shares sponsors with other highly central areas.

In [ ]:
# Centrality measures
degree_c = nx.degree_centrality(G_pol)
eigen_c = nx.eigenvector_centrality_numpy(G_pol, weight="weight")
between_c = nx.betweenness_centrality(G_pol, weight="weight")

centrality_df = pd.DataFrame({
    "Policy Area": list(G_pol.nodes()),
    "Sponsors": [G_pol.nodes[n].get("sponsor_count", 0) for n in G_pol.nodes()],
    "Degree Centrality": [degree_c[n] for n in G_pol.nodes()],
    "Eigenvector Centrality": [eigen_c[n] for n in G_pol.nodes()],
    "Betweenness Centrality": [between_c[n] for n in G_pol.nodes()],
    "Weighted Degree": [G_pol.degree(n, weight="weight") for n in G_pol.nodes()],
}).sort_values("Eigenvector Centrality", ascending=False)

fig_central = px.scatter(
    centrality_df, x="Weighted Degree", y="Eigenvector Centrality",
    size="Sponsors", hover_data=["Policy Area", "Betweenness Centrality"],
    text="Policy Area",
    title="Policy Area Centrality: Influence vs. Connectivity",
    labels={"Weighted Degree": "Weighted Degree (total shared sponsors)",
            "Eigenvector Centrality": "Eigenvector Centrality"},
)
fig_central.update_traces(textposition="top center", textfont_size=8)
fig_central.update_layout(template="plotly_white", height=500, width=800)
fig_central.show()

print("Top 10 most central policy areas (eigenvector):")
centrality_df.head(10)[["Policy Area", "Sponsors", "Eigenvector Centrality", "Weighted Degree"]]

## 3.3 Taxonomy Co-occurrence: Strategies × Harms

When legislators address a specific harm, which regulatory strategies do they pair it with? This reveals the "legislative playbook" — the harm→strategy mapping embedded in AI policy bills.

In [ ]:
# Build co-occurrence graph between Strategies and Harms
G_sh = au.build_taxonomy_cooccurrence_graph(docs_df, "Strategies", "Harms")
print(f"Strategies × Harms co-occurrence: {G_sh.number_of_nodes()} tags, {G_sh.number_of_edges()} edges")

# Extract as matrix for heatmap
strat_nodes = [n for n, d in G_sh.nodes(data=True) if d.get("group") == "Strategies"]
harm_nodes = [n for n, d in G_sh.nodes(data=True) if d.get("group") == "Harms"]
node_label = {n: d.get("label", n) for n, d in G_sh.nodes(data=True)}

matrix = pd.DataFrame(0,
                      index=sorted(node_label[n] for n in harm_nodes),
                      columns=sorted(node_label[n] for n in strat_nodes))
for u, v, d in G_sh.edges(data=True):
    ug = G_sh.nodes[u].get("group")
    if ug == "Harms":
        matrix.loc[node_label[u], node_label[v]] = d["weight"]
    else:
        matrix.loc[node_label[v], node_label[u]] = d["weight"]

# Filter to tags with meaningful signal (>0 in at least 2 cells)
matrix = matrix.loc[matrix.sum(axis=1) > 0, matrix.sum(axis=0) > 0]

fig_sh = px.imshow(
    matrix.values,
    x=[c[:35] for c in matrix.columns],
    y=[r[:40] for r in matrix.index],
    color_continuous_scale="Blues",
    labels={"color": "Co-occurrences"},
    title="Legislative Playbook: When a Harm is Addressed, Which Strategy is Used?",
    aspect="auto",
)
fig_sh.update_layout(template="plotly_white", height=400, width=1000,
                      xaxis_tickangle=-40)
fig_sh.show()

## 3.4 Strategies × Applications Co-occurrence

Which application domains receive which regulatory strategies? This reveals where oversight attention is concentrated vs. absent.

In [ ]:
# Strategies × Applications
G_sa = au.build_taxonomy_cooccurrence_graph(docs_df, "Strategies", "Applications")

strat_nodes_sa = [n for n, d in G_sa.nodes(data=True) if d.get("group") == "Strategies"]
app_nodes_sa = [n for n, d in G_sa.nodes(data=True) if d.get("group") == "Applications"]
node_label_sa = {n: d.get("label", n) for n, d in G_sa.nodes(data=True)}

matrix_sa = pd.DataFrame(0,
                         index=sorted(node_label_sa[n] for n in app_nodes_sa),
                         columns=sorted(node_label_sa[n] for n in strat_nodes_sa))
for u, v, d in G_sa.edges(data=True):
    ug = G_sa.nodes[u].get("group")
    if ug == "Applications":
        matrix_sa.loc[node_label_sa[u], node_label_sa[v]] = d["weight"]
    else:
        matrix_sa.loc[node_label_sa[v], node_label_sa[u]] = d["weight"]

matrix_sa = matrix_sa.loc[matrix_sa.sum(axis=1) > 0, matrix_sa.sum(axis=0) > 0]

fig_sa = px.imshow(
    matrix_sa.values,
    x=[c[:35] for c in matrix_sa.columns],
    y=[r[:40] for r in matrix_sa.index],
    color_continuous_scale="Greens",
    labels={"color": "Co-occurrences"},
    title="Regulatory Coverage: Which Application Domains Get Which Strategies?",
    aspect="auto",
)
fig_sa.update_layout(template="plotly_white", height=500, width=1000,
                      xaxis_tickangle=-40)
fig_sa.show()

## 3.5 Isolated vs. Integrated Policy Domains

Measure each policy area's "integration score" — what fraction of all sponsors in that area also work on at least one other area? Low values mean a siloed legislative community.

In [ ]:
# Compute integration score per policy area
cosp_with_policy = cosp_df.merge(docs_df[["AGORA ID", "Policy_Area"]], on="AGORA ID", how="left")
sponsor_areas = cosp_with_policy.groupby("Cosponsor_BioguideId")["Policy_Area"].apply(set)

area_sponsors = defaultdict(set)
for bio, areas in sponsor_areas.items():
    for a in areas:
        if pd.notna(a):
            area_sponsors[a].add(bio)

integration = {}
for area, sponsors in area_sponsors.items():
    multi_area = sum(1 for s in sponsors if len(sponsor_areas.get(s, set())) > 1)
    integration[area] = {
        "total_sponsors": len(sponsors),
        "multi_area_sponsors": multi_area,
        "integration_score": multi_area / len(sponsors) if sponsors else 0,
        "avg_areas_per_sponsor": np.mean([len(sponsor_areas.get(s, set())) for s in sponsors]),
    }

integ_df = pd.DataFrame(integration).T.sort_values("integration_score", ascending=True)
integ_df.index.name = "Policy Area"

fig_integ = px.bar(
    integ_df.reset_index(), x="integration_score", y="Policy Area",
    orientation="h",
    color="avg_areas_per_sponsor",
    color_continuous_scale="RdYlGn",
    hover_data=["total_sponsors", "multi_area_sponsors"],
    labels={"integration_score": "Integration Score (% multi-area sponsors)",
            "avg_areas_per_sponsor": "Avg Areas/Sponsor"},
    title="Policy Domain Integration: Siloed vs. Connected Legislative Communities",
)
fig_integ.update_layout(template="plotly_white", height=600, width=800)
fig_integ.show()

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Build cosponsor networks per policy area
policy_areas = docs_df['Policy_Area'].dropna().unique()
print(f"Processing {len(policy_areas)} policy areas...\n")

# Store figures for export
figs_dimred = {}

for area in sorted(policy_areas):
    # Filter docs and cosponsors for this policy area
    area_docs = docs_df[docs_df['Policy_Area'] == area]
    area_doc_ids = set(area_docs['AGORA ID'].values)
    area_cosp = cosp_df[cosp_df['AGORA ID'].isin(area_doc_ids)].copy()
    
    if len(area_cosp) < 3:  # Skip if too few cosponsors
        continue
    
    # Build sponsor-sponsor graph
    G_area = au.build_cosponsor_cosponsor_graph(area_cosp, min_shared=1)
    
    if G_area.number_of_nodes() < 2:
        continue
    
    # Extract node features: degree, betweenness, clustering coefficient
    nodes_list = list(G_area.nodes())
    features = []
    node_parties = []
    
    betweenness_map = nx.betweenness_centrality(G_area)
    for node in nodes_list:
        degree = G_area.degree(node)
        betweenness = betweenness_map.get(node, 0)
        clustering = nx.clustering(G_area, node)
        features.append([degree, betweenness, clustering])
        party = G_area.nodes[node].get('party', 'I')
        node_parties.append(party)
    
    # Dimensionality reduction with PCA
    features_scaled = StandardScaler().fit_transform(features)
    coords = PCA(n_components=2, random_state=42).fit_transform(features_scaled)
    
    # Prepare edge colors: purple for cross-party, gray for same-party
    edge_traces = []
    for u, v in G_area.edges():
        p_u = G_area.nodes[u].get('party', 'I')
        p_v = G_area.nodes[v].get('party', 'I')
        
        x0, y0 = coords[nodes_list.index(u)]
        x1, y1 = coords[nodes_list.index(v)]
        
        is_bipartisan = p_u != p_v
        color = 'rgba(128,0,128,0.3)' if is_bipartisan else 'rgba(150,150,150,0.2)'
        
        edge_traces.append((x0, x1, y0, y1, color))
    
    # Map node colors by party
    node_colors = []
    for party in node_parties:
        if party == 'D':
            node_colors.append('blue')
        elif party == 'R':
            node_colors.append('red')
        else:
            node_colors.append('gray')
    
    # Create figure
    fig = go.Figure()
    
    # Add edges
    for x0, x1, y0, y1, color in edge_traces:
        fig.add_trace(go.Scatter(
            x=[x0, x1], y=[y0, y1],
            mode='lines', line=dict(width=0.5, color=color),
            hoverinfo='none', showlegend=False
        ))
    
    # Add nodes
    for idx, node in enumerate(nodes_list):
        color = node_colors[idx]
        fig.add_trace(go.Scatter(
            x=[coords[idx, 0]], y=[coords[idx, 1]],
            mode='markers', marker=dict(size=8, color=color),
            text=f"{node}<br>Party: {node_parties[idx]}",
            hoverinfo='text', showlegend=False
        ))
    
    fig.update_layout(
        title=f"Sponsor Network (PCA on degree/betweenness/clustering): {area}",
        xaxis_title="PC 1", yaxis_title="PC 2",
        template='plotly_white', height=600, width=800,
        hovermode='closest'
    )
    
    figs_dimred[area] = fig
    fig.show()
    print(f"  {area}: {G_area.number_of_nodes()} sponsors, {G_area.number_of_edges()} edges")

print(f"\nGenerated {len(figs_dimred)} dimensionality reduction visualizations")

## 3.6 Sponsor Network Dimensionality Reduction per Policy Area

Reduce cosponsor-cosponsor networks to 2D for each policy area using PCA, colored by party affiliation: blue=Democrat, red=Republican, purple=bipartisan edges.

## 3.6 Exports

In [ ]:
# Save graphs and figures
au.save_graph(G_pol, "policy_cooccurrence_graph")
au.save_graph(G_sh, "strategies_harms_cooccurrence")
au.save_graph(G_sa, "strategies_applications_cooccurrence")
au.save_df(centrality_df, "policy_area_centrality")
au.save_df(integ_df, "policy_integration_scores")

fig_net.write_html(str(au.OUTPUTS_DIR / "fig_policy_network.html"))
fig_central.write_html(str(au.OUTPUTS_DIR / "fig_policy_centrality.html"))
fig_sh.write_html(str(au.OUTPUTS_DIR / "fig_strategies_harms.html"))
fig_sa.write_html(str(au.OUTPUTS_DIR / "fig_strategies_applications.html"))
fig_integ.write_html(str(au.OUTPUTS_DIR / "fig_policy_integration.html"))

# Save dimensionality reduction figures per policy area
for area, fig in figs_dimred.items():
    safe_name = area.replace(" ", "_").replace("/", "_").lower()
    fig.write_html(str(au.OUTPUTS_DIR / f"fig_sponsor_dimred_{safe_name}.html"))

print("Module 3 exports saved to notebooks/outputs/")